# Visual Exploration of Graph Neural Networks in Your Computational Notebooks

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import sys, os
sys.path.append(os.path.abspath("src"))

## Visualizing Graphs with Dual Views

- **API**: GraphVisualizer
- **Parameter**: dataFile (the file path of the graph data that you want to visualize). 

In [3]:
from gnn_exp import GraphVisualizer
w = GraphVisualizer()
w.add_data(dataFile="test_data/input_graph0.json")
w

Loading JSON data from: /Users/harrylu_mac/gnn-explorer/test_data/input_graph0.json
graphData: dict_keys(['x', 'edge_index', 'edge_attr', 'y', 'batch']), loaded, path: /Users/harrylu_mac/gnn-explorer/src/gnn_exp/static.


GraphVisualizer(graphData={'x': [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.…

### Using GPU Renderers with Dual Views

`GraphVisualizer` defaults to `renderer="svg"`. To request a GPU-backed renderer, pass `renderer="webgl"`, `renderer="webgpu"`, or `renderer="auto"` when creating the widget. `auto` tries `webgpu -> webgl -> svg`, and `effectiveRenderer` reflects the backend that actually becomes active after browser capability checks and fallback.

In [ ]:
gpu_w = GraphVisualizer(renderer="auto")
gpu_w.add_data(dataFile="test_data/input_graph0.json")
gpu_w

### Rendering a Synthetic Large Graph with WebGL

The example below fabricates a larger connected graph directly in the notebook and renders it with `renderer="webgl"`. The matrix view scales with `num_nodes ** 2`, so start with the default size first; increase `LARGE_NODE_COUNT` and `LARGE_EXTRA_EDGES` when you specifically want a heavier GPU stress test. For large graphs, matrix axis labels are hidden automatically to keep the view readable.

In [ ]:
import random

LARGE_NODE_COUNT = 400
LARGE_EXTRA_EDGES = 1200

def make_synthetic_graph(num_nodes=LARGE_NODE_COUNT, extra_edges=LARGE_EXTRA_EDGES, feature_width=7, seed=7):
    rng = random.Random(seed)
    edges = set()

    for node in range(num_nodes):
        source = node
        target = (node + 1) % num_nodes
        edges.add((min(source, target), max(source, target)))

    while len(edges) < num_nodes + extra_edges:
        source = rng.randrange(num_nodes)
        target = rng.randrange(num_nodes)
        if source == target:
            continue
        edges.add((min(source, target), max(source, target)))

    sources, targets = zip(*sorted(edges))
    features = []
    for node in range(num_nodes):
        feature = [0.0] * feature_width
        feature[node % feature_width] = 1.0
        features.append(feature)

    return {
        "x": features,
        "edge_index": [list(sources), list(targets)],
        "y": [node % 2 for node in range(num_nodes)],
        "batch": [0] * num_nodes,
    }

large_graph_data = make_synthetic_graph()
print(
    f"Synthetic graph: {len(large_graph_data['x'])} nodes, "
    f"{len(large_graph_data['edge_index'][0])} edges"
)

large_gpu_w = GraphVisualizer(graphData=large_graph_data, renderer="webgl")
large_gpu_w

## Visualizing Large Graphs with Subgraph Dual Views

- **API**:
- GraphVisualizer (class)
- subgraph_hoop_visualizer (API)
- multiple_subgraph_hoop_visualizer (API)
- **Parameter**:
- dataFile (the file path of the graph data that you want to visualize).
- hubNode (the hub node that we want to visualize)
- hubNodes (an array of hub nodes that need to be visualize)
- hoopNum (the hoop number we used for partition)

### Visualizing Large Graphs with Hoop-based Extraction (Single Hub Node)

In [4]:
sub_w = GraphVisualizer()
sub_w.add_data(dataFile="test_data/twitch.json")
sub_w.subgraph_hoop_visualizer(hubNode=0, hoopNum=3)
sub_w

Loading JSON data from: /Users/harrylu_mac/gnn-explorer/test_data/twitch.json
graphData: dict_keys(['x', 'edge_index', 'y', 'batch']), loaded, path: /Users/harrylu_mac/gnn-explorer/src/gnn_exp/static.
Updated to 3-hop subgraph centered at 0.


GraphVisualizer(graphData={'x': [[-0.23666860163211823, -0.23071609437465668, -0.16054700314998627, -0.1982001…

### Visualizing Large Graphs with Hoop-based Extraction (Multiple Hub Nodes)

In [5]:
mul_sub_w = GraphVisualizer()
mul_sub_w.add_data(dataFile="test_data/twitch.json")
mul_sub_w.multiple_subgraph_hoop_visualizer(hubNodes=[0, 1], hoopNum=1)
mul_sub_w

Loading JSON data from: /Users/harrylu_mac/gnn-explorer/test_data/twitch.json
graphData: dict_keys(['x', 'edge_index', 'y', 'batch']), loaded, path: /Users/harrylu_mac/gnn-explorer/src/gnn_exp/static.
Updated to 1-hop subgraphs centered at [0, 1].


GraphVisualizer(graphData={'x': [[-0.23666860163211823, -0.23071609437465668, -0.16054700314998627, -0.1982001…

## Editing Graph Structures through GraphEditor

- **API**: GraphEditor
- **Parameter**: dataFile (the file path of the graph data that you want to visualize). 

In [6]:
from gnn_exp import GraphEditor
editor = GraphEditor()
editor.add_data(dataFile="test_data/karate_dataset.json")
editor

Exposing file to browser: /files/test_data/karate_dataset.json


GraphEditor()

### Exporting Data from GraphEditor through DataBridge

In [7]:
import json
full_path = "test_data/karate_dataset.json"
with open(full_path, "r") as f:
    data_before = json.load(f)
len(data_before['x'])

34

In [8]:
editor.export_data_to_json("test_data/test_new_data.json")

Graph data exported to test_data/test_new_data.json


## Visualizing Model Intermediate Features through Matrix View (WIP)

- **API**: GNNVisualizer

### Constructuring a Graph Neural Network using PyTorch Geometric

**You must register activation layer when create an GNN model. **

In [9]:
import os
import torch

from torch_geometric.datasets import KarateClub
dataset = KarateClub()
data = dataset[0]
edge_index = data.edge_index
print(edge_index.t())

import torch.nn as nn
from torch.nn import Linear
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self, dataset):
        super().__init__()
        torch.manual_seed(1234)

        self.conv1 = GCNConv(dataset.num_features, 4)
        self.act1  = nn.Tanh()

        self.conv2 = GCNConv(4, 4)
        self.act2  = nn.Tanh()

        self.conv3 = GCNConv(4, 4)
        self.act3  = nn.Tanh()

        self.conv4 = GCNConv(4, 2)
        self.act4  = nn.Tanh()

        self.classifier = Linear(2, dataset.num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        outputs = {}

        h = self.conv1(x, edge_index)
        outputs["conv1_linear"] = h
        h = self.act1(h)
        outputs["conv1_act"] = h

        h = self.conv2(h, edge_index)
        outputs["conv2_linear"] = h
        h = self.act2(h)
        outputs["conv2_act"] = h

        h = self.conv3(h, edge_index)
        outputs["conv3_linear"] = h
        h = self.act3(h)
        outputs["conv3_act"] = h

        h = self.conv4(h, edge_index)
        outputs["conv4_linear"] = h
        h = self.act4(h)
        outputs["conv4_act"] = h

        out = self.classifier(h)
        outputs["logits"] = out

        prob = self.softmax(out)
        outputs["final"] = prob

        return outputs

model = GCN(dataset)

tensor([[ 0,  1],
        [ 0,  2],
        [ 0,  3],
        [ 0,  4],
        [ 0,  5],
        [ 0,  6],
        [ 0,  7],
        [ 0,  8],
        [ 0, 10],
        [ 0, 11],
        [ 0, 12],
        [ 0, 13],
        [ 0, 17],
        [ 0, 19],
        [ 0, 21],
        [ 0, 31],
        [ 1,  0],
        [ 1,  2],
        [ 1,  3],
        [ 1,  7],
        [ 1, 13],
        [ 1, 17],
        [ 1, 19],
        [ 1, 21],
        [ 1, 30],
        [ 2,  0],
        [ 2,  1],
        [ 2,  3],
        [ 2,  7],
        [ 2,  8],
        [ 2,  9],
        [ 2, 13],
        [ 2, 27],
        [ 2, 28],
        [ 2, 32],
        [ 3,  0],
        [ 3,  1],
        [ 3,  2],
        [ 3,  7],
        [ 3, 12],
        [ 3, 13],
        [ 4,  0],
        [ 4,  6],
        [ 4, 10],
        [ 5,  0],
        [ 5,  6],
        [ 5, 10],
        [ 5, 16],
        [ 6,  0],
        [ 6,  4],
        [ 6,  5],
        [ 6, 16],
        [ 7,  0],
        [ 7,  1],
        [ 7,  2],
        [ 

In [10]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Define optimizer.

def train(data):
    optimizer.zero_grad()  # Clear gradients.
    outputs = model(data.x, data.edge_index)  # Perform a single forward pass.
    out = outputs["final"]
    h = outputs["conv4_act"]
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    optimizer.step()  # Update parameters based on gradients.
    return loss, h

for epoch in range(401):
    loss, h = train(data)
print("training is finished")

training is finished


In [11]:
outputs = model(data.x, data.edge_index)

print(f"conv1 shape: {list(outputs['conv1_act'].shape)}")
print(f"conv2 shape: {list(outputs['conv2_act'].shape)}")
print(f"conv3 shape: {list(outputs['conv3_act'].shape)}")
print(f"conv4 shape: {list(outputs['conv4_act'].shape)}")
print(f"final shape: {list(outputs['final'].shape)}")

conv1 shape: [34, 4]
conv2 shape: [34, 4]
conv3 shape: [34, 4]
conv4 shape: [34, 2]
final shape: [34, 4]


### Model Patching Automation

In [12]:
print(model)

GCN(
  (conv1): GCNConv(34, 4)
  (act1): Tanh()
  (conv2): GCNConv(4, 4)
  (act2): Tanh()
  (conv3): GCNConv(4, 4)
  (act3): Tanh()
  (conv4): GCNConv(4, 2)
  (act4): Tanh()
  (classifier): Linear(in_features=2, out_features=4, bias=True)
  (softmax): Softmax(dim=1)
)


In [13]:
layers_info = []

for name, module in model.named_children():
    layers_info.append({
        "name": name,
        "type": module.__class__.__name__,
        "params": module
    })

layers_info

[{'name': 'conv1', 'type': 'GCNConv', 'params': GCNConv(34, 4)},
 {'name': 'act1', 'type': 'Tanh', 'params': Tanh()},
 {'name': 'conv2', 'type': 'GCNConv', 'params': GCNConv(4, 4)},
 {'name': 'act2', 'type': 'Tanh', 'params': Tanh()},
 {'name': 'conv3', 'type': 'GCNConv', 'params': GCNConv(4, 4)},
 {'name': 'act3', 'type': 'Tanh', 'params': Tanh()},
 {'name': 'conv4', 'type': 'GCNConv', 'params': GCNConv(4, 2)},
 {'name': 'act4', 'type': 'Tanh', 'params': Tanh()},
 {'name': 'classifier',
  'type': 'Linear',
  'params': Linear(in_features=2, out_features=4, bias=True)},
 {'name': 'softmax', 'type': 'Softmax', 'params': Softmax(dim=1)}]

In [14]:
### Model Hook

model.eval()
layer_outputs = {}

def save_output(name):
    def hook(module, input, output):
        layer_outputs[name] = output.detach()
    return hook

# register hook
hooks = []

for name, module in model.named_children():
    h = module.register_forward_hook(save_output(name))
    hooks.append(h)

with torch.no_grad():
    out = model(data.x, edge_index)

layer_outputs.keys()

dict_keys(['conv1', 'act1', 'conv2', 'act2', 'conv3', 'act3', 'conv4', 'act4', 'classifier', 'softmax'])

In [15]:
layer_outputs['classifier']

tensor([[-5.2745e+00,  4.8138e+00,  5.5536e-02,  1.1573e-01],
        [-6.4362e-01,  1.7485e-02,  4.0510e+00, -4.0727e+00],
        [ 2.3421e-01, -8.5776e-01,  3.9351e+00, -3.9168e+00],
        [-8.4011e-01,  2.2565e-01,  3.7617e+00, -3.7647e+00],
        [-9.3810e-02, -2.2454e-01, -3.8996e+00,  4.5935e+00],
        [-1.0093e-01, -2.1755e-01, -3.8958e+00,  4.5892e+00],
        [-1.0093e-01, -2.1755e-01, -3.8958e+00,  4.5892e+00],
        [-9.5706e-02, -5.1995e-01,  3.7504e+00, -3.7271e+00],
        [ 4.9273e+00, -5.4226e+00,  3.6986e-01,  1.2035e-01],
        [ 4.9328e+00, -5.4280e+00,  3.6623e-01,  1.2449e-01],
        [-9.3807e-02, -2.2454e-01, -3.8996e+00,  4.5935e+00],
        [-1.0663e-01, -2.1277e-01, -3.8717e+00,  4.5628e+00],
        [-2.8476e+00,  2.4065e+00, -5.8752e-01,  8.9759e-01],
        [ 8.4137e-01, -1.4447e+00,  3.3810e+00, -3.2935e+00],
        [ 4.9554e+00, -5.4500e+00,  3.4903e-01,  1.4397e-01],
        [ 4.9555e+00, -5.4500e+00,  3.4898e-01,  1.4403e-01],
        

In [16]:
model.state_dict().keys()

odict_keys(['conv1.bias', 'conv1.lin.weight', 'conv2.bias', 'conv2.lin.weight', 'conv3.bias', 'conv3.lin.weight', 'conv4.bias', 'conv4.lin.weight', 'classifier.weight', 'classifier.bias'])

In [17]:
model.state_dict()['classifier.bias']

tensor([-0.1239, -0.3523,  0.1643,  0.1724])

### Visualizing a Graph Neural Network in Computational Notebooks
- **API**: GNNVisualizer
- **Parameters**:
- graphFIle: input graph JSON file.
- weightFile: model weight JSON file.
- modelInfo: the intermediate layer outputs and model architecture information.

**TODO**

- implement GraphSAGE and GAT layer inners
- final assemble - compose the example
- add computation symbol to the vis
- implement node-link view
- compose the document
- multiple instances support

#### Node Task Visualization

In [18]:
from gnn_exp import GNNVisualizer
model_w = GNNVisualizer(graphData={}, graphPath="test_data/test_new_data.json", intmData={})

In [19]:
model_w.add_model(data, model,False, None, [[12, 18]], "node")
model_w

mode: node
Python: queries changed to: [[12, 18]], type: <class 'list'>
Queries set to: [[12, 18]]
check act0: [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]
modelInfo: dict_keys(['conv1

GNNVisualizer(graphData={'x': [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0…

In [20]:
model_e = GNNVisualizer(graphData={}, graphPath="test_data/test_new_data.json", intmData={})
model_e.add_model(data, model,False, None, [[12, 18]], "edge")
model_e

mode: edge
Python: queries changed to: [[12, 18]], type: <class 'list'>
Queries set to: [[12, 18]]
check act0: [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]
modelInfo: dict_keys(['conv1

GNNVisualizer(graphData={'x': [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0…

#### Link Task Visualization

In [21]:
class DotProductDecoder(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, z, edge_label_index):
        src, dst = edge_label_index
        score = (z[src] * z[dst]).sum(dim=1)
        return score

class GCNForLinkPred(torch.nn.Module):
    def __init__(self, dataset):
        super().__init__()
        torch.manual_seed(1234)

        self.conv1 = GCNConv(dataset.num_features, 4)
        self.act1  = nn.Tanh()

        self.conv2 = GCNConv(4, 4)
        self.act2  = nn.Tanh()

        self.conv3 = GCNConv(4, 4)
        self.act3  = nn.Tanh()

        self.conv4 = GCNConv(4, 2)
        self.act4  = nn.Tanh()

        self.decoder = DotProductDecoder()

    def encode(self, x, edge_index):
        outputs = {}

        h = self.conv1(x, edge_index)
        outputs["conv1_linear"] = h
        h = self.act1(h)
        outputs["conv1_act"] = h

        h = self.conv2(h, edge_index)
        outputs["conv2_linear"] = h
        h = self.act2(h)
        outputs["conv2_act"] = h

        h = self.conv3(h, edge_index)
        outputs["conv3_linear"] = h
        h = self.act3(h)
        outputs["conv3_act"] = h

        h = self.conv4(h, edge_index)
        outputs["conv4_linear"] = h
        h = self.act4(h)
        outputs["conv4_act"] = h

        outputs["node_embedding"] = h
        return h, outputs

    def forward(self, x, edge_index, edge_label_index):
        z, outputs = self.encode(x, edge_index)

        edge_score = self.decoder(z, edge_label_index)
        outputs["edge_score"] = edge_score

        return outputs

In [22]:
link_model = GCNForLinkPred(dataset)
print(link_model)

GCNForLinkPred(
  (conv1): GCNConv(34, 4)
  (act1): Tanh()
  (conv2): GCNConv(4, 4)
  (act2): Tanh()
  (conv3): GCNConv(4, 4)
  (act3): Tanh()
  (conv4): GCNConv(4, 2)
  (act4): Tanh()
  (decoder): DotProductDecoder()
)


In [23]:
from torch_geometric.utils import negative_sampling

def build_edge_label_data(data, num_neg_per_pos=1, undirected=True):
    edge_index = data.edge_index
    num_nodes = data.num_nodes
    pos_edge_index = edge_index
    neg_edge_index = negative_sampling(
        edge_index=edge_index,
        num_nodes=num_nodes,
        num_neg_samples=pos_edge_index.size(1) * num_neg_per_pos,
        method="sparse"
    )
    edge_label_index = torch.cat(
        [pos_edge_index, neg_edge_index],
        dim=1
    )
    edge_label = torch.cat(
        [
            torch.ones(pos_edge_index.size(1)),
            torch.zeros(neg_edge_index.size(1))
        ],
        dim=0
    )
    data.edge_label_index = edge_label_index
    data.edge_label = edge_label
    return data

In [24]:
data = build_edge_label_data(data)

In [25]:
def link_forward(model, data):
    model(
        data.x,
        data.edge_index,
        data.edge_label_index
    )

In [26]:
model_link_w = GNNVisualizer(graphData={}, graphPath="test_data/test_new_data.json", intmData={})
model_link_w.add_model(data, link_model, link_forward)
model_link_w

TraitError: The 'subgraphSample' trait of a GNNVisualizer instance expected a boolean, not the function 'link_forward'.

#### Graph Task Visualization

### Visualizing Different Graph Neural Networks using GNNVisualizer

#### Visualizing a Graph Attention Network

#### Visualizing a Graph Sampling and Aggregation

### Visualizing Computational Graph for a Subgraph
- **API**: NodeComputationalGraph